# Woche 11 · Context Engineering und Memory
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Ein Instandhalter spricht eine ganze Schicht mit dem Agenten. Mit jeder Frage wächst der Verlauf, und jede Anfrage schickt alles erneut mit. Das kostet, und der Agent verliert Wichtiges aus dem Blick. Heute behandelt ihr den Kontext als **knappes Budget**:

1. **Messen:** Woraus besteht der Kontext, und wie wächst er?
2. **Kompaktieren:** alte Runden durch eine Zusammenfassung ersetzen
3. **Langzeit-Memory:** Notizen über das Gespräch hinaus, mit `remember` und `recall`, und mit Datenschutz
4. **Prompt Caching:** gleichbleibende Teile zum Zehntel des Preises

Pflicht: TODO 1–6, Ziel: TODO 7–8. Toolbox-Schlüssel nötig, etwa 150 Anfragen pro Person.

In [ ]:
%pip install -q openai==3.24.0 rank-bm25==0.2.2

In [ ]:
import json, re, os, math, datetime, time, copy
import matplotlib.pyplot as plt
from openai import OpenAI
from rank_bm25 import BM25Okapi

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL = "kit.mistral-small-4-119b-a8b"
HEUTE = datetime.date.today()
# Kommerzielle Referenzpreise in Dollar pro 1 Mio. Tokens, mit Prompt Caching (wie Claude Haiku 4.5 laut Anthropic-Preisseite).
# In der KI-Toolbox zahlt ihr nichts. Die Preise zeigen, was dieselbe Nutzung bei einem kommerziellen Anbieter kosten würde.
PREISE = {"input": 1.00, "cache_write": 1.25, "cache_read": 0.10, "output": 5.00}
print("Schlüssel geladen:", bool(API_KEY))

## Daten, Tools und ein Gesprächs-Agent (vorgegeben, einfach ausführen)

Wie in Woche 10. Neu ist die Klasse `Gespraech`: Sie behält den Verlauf über mehrere Fragen und protokolliert pro Frage die Tokens, getrennt nach normalem Input, Cache-Schreiben, Cache-Lesen und Output.

In [ ]:
MASCHINEN = [
    {'id': 'M-01', 'name': 'CNC-Fräszentrum 1', 'typ': 'Kessler KFZ 800', 'baujahr': 2018, 'standort': 'Halle 1, Platz 4', 'status': 'in Betrieb', 'betriebsstunden': 28450, 'letzte_wartung': '2026-08-12', 'service': 'Kessler Service, SV-1180'},
    {'id': 'M-02', 'name': 'CNC-Fräszentrum 2', 'typ': 'Kessler KFZ 800 plus', 'baujahr': 2021, 'standort': 'Halle 1, Platz 5', 'status': 'in Betrieb', 'betriebsstunden': 14120, 'letzte_wartung': '2026-09-02', 'service': 'Kessler Service, SV-1180'},
    {'id': 'M-03', 'name': 'CNC-Drehmaschine', 'typ': 'Hartwig TD 400', 'baujahr': 2015, 'standort': 'Halle 1, Platz 9', 'status': 'eingeschränkt', 'betriebsstunden': 41980, 'letzte_wartung': '2026-07-28', 'service': 'kein Servicevertrag'},
    {'id': 'M-04', 'name': 'Rundschleifmaschine', 'typ': 'Brenner RS 600', 'baujahr': 2019, 'standort': 'Halle 2, Platz 2', 'status': 'in Betrieb', 'betriebsstunden': 22310, 'letzte_wartung': '2026-09-15', 'service': 'kein Servicevertrag'},
    {'id': 'M-05', 'name': 'Roboter-Schweißzelle', 'typ': 'Zelle mit Varion 500', 'baujahr': 2020, 'standort': 'Halle 2, Platz 6', 'status': 'in Betrieb', 'betriebsstunden': 19870, 'letzte_wartung': '2026-09-09', 'service': 'SV-2045'},
    {'id': 'M-06', 'name': 'Montagelinie A', 'typ': 'Eigenbau, Umbau 2023', 'baujahr': 2017, 'standort': 'Halle 2, Bereich M', 'status': 'in Betrieb', 'betriebsstunden': 30560, 'letzte_wartung': '2026-08-30', 'service': 'kein Servicevertrag'},
    {'id': 'M-07', 'name': 'Prüfstand', 'typ': 'Druckprüfeinheit mit Messrechner', 'baujahr': 2022, 'standort': 'Halle 2, Bereich P', 'status': 'in Betrieb', 'betriebsstunden': 8740, 'letzte_wartung': '2026-06-20', 'service': 'Kalibrierung über QS'},
    {'id': 'M-08', 'name': 'Teilewaschanlage', 'typ': 'Durchlaufwaschanlage', 'baujahr': 2016, 'standort': 'Halle 1, Platz 12', 'status': 'in Betrieb', 'betriebsstunden': 33900, 'letzte_wartung': '2026-09-20', 'service': 'kein Servicevertrag'},
]
WARTUNGSHISTORIE = [
    ('M-01', '2026-08-12', 'Hydrauliköl gewechselt, Filter getauscht', 'Instandhaltung'),
    ('M-01', '2026-06-03', 'Abstreifer Führungsbahn X erneuert (KFZ-8810)', 'Instandhaltung'),
    ('M-01', '2026-03-18', 'Spindelrundlauf gemessen: 3 µm, in Ordnung', 'Instandhaltung'),
    ('M-02', '2026-09-02', 'Greifer des Werkzeugwechslers gereinigt, Federn getauscht (KFZ-9120)', 'Instandhaltung'),
    ('M-02', '2026-07-01', 'Messtaster kalibriert', 'Instandhaltung'),
    ('M-02', '2026-04-14', 'Kugelgewindetriebe geprüft, kein Spiel', 'Instandhaltung'),
    ('M-03', '2026-07-28', 'Spindellager nachgeschmiert (NW-LX2, 8 g je Lager)', 'Instandhaltung'),
    ('M-03', '2026-07-28', 'Spindelrundlauf gemessen: 7 µm, nahe am Grenzwert 8 µm', 'Instandhaltung'),
    ('M-03', '2026-06-30', 'Spannfutter zerlegt, gereinigt, Backen gefettet', 'Instandhaltung'),
    ('M-03', '2026-05-19', 'Spindellager nachgeschmiert (NW-LX2, 8 g je Lager)', 'Instandhaltung'),
    ('M-03', '2026-02-10', 'Hydrauliköl gewechselt, Rücklauffilter getauscht (HTD-3310)', 'Instandhaltung'),
    ('M-04', '2026-09-15', 'Neue Schleifscheibe montiert, Klangprobe in Ordnung, ausgewuchtet', 'Instandhaltung'),
    ('M-04', '2026-08-11', 'Wuchteinrichtung geprüft', 'Instandhaltung'),
    ('M-04', '2026-06-23', 'Schleifspindel auf Lagergeräusche geprüft, unauffällig', 'Instandhaltung'),
    ('M-05', '2026-09-09', 'Hydraulik der Spannvorrichtungen geprüft, kleine Leckage an Zylinder 2 abgedichtet', 'Instandhaltung'),
    ('M-05', '2026-08-05', 'Schweißstromquelle getauscht nach Ausfall', 'Servicepartner SV-2045'),
    ('M-05', '2026-05-12', 'Kühlflüssigkeit des Brenners ersetzt', 'Instandhaltung'),
    ('M-06', '2026-08-30', 'Kettenspannung Fördersystem nachgestellt', 'Instandhaltung'),
    ('M-06', '2026-07-14', 'Datensicherung der SPS nach Programmänderung', 'Instandhaltung'),
    ('M-06', '2026-06-02', 'Zentralschmierung aufgefüllt (NW-K100)', 'Instandhaltung'),
    ('M-07', '2026-06-20', 'Kalibrierung der Drucksensoren durch Dienstleister', 'Qualitätssicherung'),
    ('M-07', '2025-10-02', 'Elektrische Prüfung nach DGUV Vorschrift 3', 'Elektrofachkraft'),
    ('M-08', '2026-09-20', 'Waschbad gewechselt, Tank gereinigt', 'Instandhaltung'),
    ('M-08', '2026-08-24', 'Waschbad gewechselt, Tank gereinigt', 'Instandhaltung'),
    ('M-08', '2026-04-07', 'Heizstab entkalkt', 'Instandhaltung'),
]
TICKETS = [
    {'ticket': 'S-001', 'datum': '2026-10-05', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 45, 'status': 'geschlossen', 'beschreibung': 'Spindel läuft unrund, Lager nachgestellt'},
    {'ticket': 'S-002', 'datum': '2026-10-05', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'elektrisch', 'prioritaet': 'hoch', 'stillstand_min': 20, 'status': 'geschlossen', 'beschreibung': 'Not-Aus durch defekten Türschalter ausgelöst'},
    {'ticket': 'S-003', 'datum': '2026-10-05', 'schicht': 'Nacht', 'maschine': 'M-06', 'kategorie': 'Material', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Fehlende Gehäuseteile an Station 3'},
    {'ticket': 'S-004', 'datum': '2026-10-06', 'schicht': 'Früh', 'maschine': 'M-05', 'kategorie': 'elektrisch', 'prioritaet': 'hoch', 'stillstand_min': 240, 'status': 'geschlossen', 'beschreibung': 'Schweißstromquelle ausgefallen, Tausch durch Instandhaltung'},
    {'ticket': 'S-005', 'datum': '2026-10-06', 'schicht': 'Spät', 'maschine': 'M-03', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 60, 'status': 'geschlossen', 'beschreibung': 'CNC-Programm nach Update nicht ladbar'},
    {'ticket': 'S-006', 'datum': '2026-10-06', 'schicht': 'Spät', 'maschine': 'M-02', 'kategorie': 'mechanisch', 'prioritaet': 'niedrig', 'stillstand_min': 25, 'status': 'geschlossen', 'beschreibung': 'Werkzeugwechsler klemmt'},
    {'ticket': 'S-007', 'datum': '2026-10-07', 'schicht': 'Früh', 'maschine': 'M-04', 'kategorie': 'Bedienung', 'prioritaet': 'niedrig', 'stillstand_min': 15, 'status': 'geschlossen', 'beschreibung': 'Falsches Schleifprogramm gewählt'},
    {'ticket': 'S-008', 'datum': '2026-10-07', 'schicht': 'Nacht', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Spänestau im Förderer'},
    {'ticket': 'S-009', 'datum': '2026-10-07', 'schicht': 'Nacht', 'maschine': 'M-07', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 50, 'status': 'geschlossen', 'beschreibung': 'Prüfsoftware hängt beim Messlauf'},
    {'ticket': 'S-010', 'datum': '2026-10-08', 'schicht': 'Früh', 'maschine': 'M-06', 'kategorie': 'Material', 'prioritaet': 'mittel', 'stillstand_min': 40, 'status': 'geschlossen', 'beschreibung': 'Schraubenzuführung leer'},
    {'ticket': 'S-011', 'datum': '2026-10-08', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 70, 'status': 'geschlossen', 'beschreibung': 'Kühlmittelpumpe undicht'},
    {'ticket': 'S-012', 'datum': '2026-10-08', 'schicht': 'Spät', 'maschine': 'M-08', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 30, 'status': 'geschlossen', 'beschreibung': 'Heizregister der Waschanlage ohne Funktion'},
    {'ticket': 'S-013', 'datum': '2026-10-09', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 90, 'status': 'geschlossen', 'beschreibung': 'Spindel erneut unrund, Lagerwechsel geplant'},
    {'ticket': 'S-014', 'datum': '2026-10-09', 'schicht': 'Nacht', 'maschine': 'M-05', 'kategorie': 'Bedienung', 'prioritaet': 'mittel', 'stillstand_min': 30, 'status': 'geschlossen', 'beschreibung': 'Schweißdraht falsch eingefädelt'},
    {'ticket': 'S-015', 'datum': '2026-10-12', 'schicht': 'Früh', 'maschine': 'M-02', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 55, 'status': 'geschlossen', 'beschreibung': 'Achsantrieb Y meldet Überstrom'},
    {'ticket': 'S-016', 'datum': '2026-10-12', 'schicht': 'Spät', 'maschine': 'M-06', 'kategorie': 'Bedienung', 'prioritaet': 'hoch', 'stillstand_min': 10, 'status': 'geschlossen', 'beschreibung': 'Lichtvorhang beim Teilewechsel unterbrochen'},
    {'ticket': 'S-017', 'datum': '2026-10-13', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 40, 'status': 'geschlossen', 'beschreibung': 'Sensor am Spannfutter liefert kein Signal'},
    {'ticket': 'S-018', 'datum': '2026-10-13', 'schicht': 'Nacht', 'maschine': 'M-04', 'kategorie': 'mechanisch', 'prioritaet': 'hoch', 'stillstand_min': 65, 'status': 'geschlossen', 'beschreibung': 'Schleifscheibe gebrochen, Austausch'},
    {'ticket': 'S-019', 'datum': '2026-10-14', 'schicht': 'Früh', 'maschine': 'M-07', 'kategorie': 'Software', 'prioritaet': 'niedrig', 'stillstand_min': 20, 'status': 'geschlossen', 'beschreibung': 'Messprotokoll wird nicht gespeichert'},
    {'ticket': 'S-020', 'datum': '2026-10-14', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'Material', 'prioritaet': 'niedrig', 'stillstand_min': 25, 'status': 'geschlossen', 'beschreibung': 'Rohteile mit Übermaß angeliefert'},
    {'ticket': 'S-021', 'datum': '2026-10-15', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 55, 'status': 'offen', 'beschreibung': 'Spindelgeräusch, Lagerwechsel fürs Wochenende eingeplant'},
    {'ticket': 'S-022', 'datum': '2026-10-15', 'schicht': 'Spät', 'maschine': 'M-08', 'kategorie': 'mechanisch', 'prioritaet': 'hoch', 'stillstand_min': 45, 'status': 'geschlossen', 'beschreibung': 'Förderband der Waschanlage gerissen'},
    {'ticket': 'S-023', 'datum': '2026-10-16', 'schicht': 'Nacht', 'maschine': 'M-02', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Werkzeugdaten nach Neustart verloren'},
    {'ticket': 'S-024', 'datum': '2026-10-16', 'schicht': 'Spät', 'maschine': 'M-06', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 75, 'status': 'offen', 'beschreibung': 'Schrauber 2 liefert falsches Drehmoment'},
]
SENSORDATEN = {
    'M-01': [('Spindeltemperatur', 41, '°C', 70, 'max'), ('Vibration Spindel', 1.9, 'mm/s', 4.5, 'max'), ('Hydraulikdruck', 62, 'bar', 55, 'min')],
    'M-02': [('Spindeltemperatur', 44, '°C', 70, 'max'), ('Vibration Spindel', 2.3, 'mm/s', 4.5, 'max'), ('Hydraulikdruck', 60, 'bar', 55, 'min')],
    'M-03': [('Spindeltemperatur', 63, '°C', 70, 'max'), ('Vibration Spindel', 4.8, 'mm/s', 4.5, 'max'), ('Hydraulikdruck', 41, 'bar', 38, 'min')],
    'M-04': [('Spindeltemperatur', 38, '°C', 65, 'max'), ('Vibration Schleifspindel', 1.1, 'mm/s', 2.8, 'max'), ('Kühlmitteltemperatur', 22, '°C', 30, 'max')],
    'M-05': [('Kühlmitteltemperatur Brenner', 68, '°C', 60, 'max'), ('Schutzgas Flaschendruck', 74, 'bar', 10, 'min'), ('Hydraulikdruck Spannvorrichtung', 95, 'bar', 90, 'min')],
    'M-06': [('Kettenspannung', 0.92, 'kN', 0.8, 'min'), ('Taktzeit', 48, 's', 55, 'max'), ('Druckluft', 6.1, 'bar', 5.5, 'min')],
    'M-07': [('Temperatur Schaltschrank', 58, '°C', 45, 'max'), ('Drehzahl Schaltschranklüfter', 0, 'U/min', 1500, 'min'), ('Druckluft', 6.0, 'bar', 5.5, 'min')],
    'M-08': [('Temperatur Waschlauge', 52, '°C', 58, 'min'), ('Füllstand Waschtank', 84, '%', 70, 'min'), ('Förderbandgeschwindigkeit', 1.2, 'm/min', 1.0, 'min')],
}
HANDBUCH = '''# Wartungshandbuch Nordwerk GmbH, Werk Karlsruhe (Auszug)

Dokument WH-2026-03 · Stand: September 2026 · Herausgeber: Abteilung Instandhaltung · Dieses Dokument ist fiktiv und dient ausschließlich Schulungszwecken.

## 1 Allgemeines

### 1.1 Geltungsbereich und Aufbau

Dieses Handbuch gilt für alle Fertigungs- und Prüfanlagen in den Hallen 1 und 2 des Werks Karlsruhe. Es richtet sich an Maschinenbedienerinnen und -bediener, Schichtleitungen und das Instandhaltungsteam. Kapitel 1 regelt Zuständigkeiten, Sicherheit und Dokumentation. Kapitel 2 beschreibt Betriebsstoffe, die an mehreren Maschinen verwendet werden. Kapitel 3 enthält für jede Maschine eine Kurzbeschreibung, die Wartungsintervalle, typische Störungen und wichtige Ersatzteile. Kapitel 4 regelt das Ersatzteillager, Kapitel 5 enthält Fehlercodes und Kontakte.

Die Herstellerdokumentation der einzelnen Maschinen bleibt verbindlich. Bei Widersprüchen gilt die Herstellerangabe, und die Instandhaltung ist zu informieren, damit dieses Handbuch angepasst werden kann. Änderungen am Handbuch gibt die Leitung Instandhaltung frei. Die jeweils gültige Fassung liegt im Intranet. Ausgedruckte Exemplare an den Maschinen sind nach jeder Änderung auszutauschen.

### 1.2 Zuständigkeiten und Eskalation

Für die tägliche Pflege und die Sichtprüfungen ist die Bedienerin oder der Bediener der Maschine zuständig. Wartungsarbeiten mit Werkzeug, das Öffnen von Schaltschränken und alle Arbeiten an Hydraulik, Druckluftverteilung und Elektrik führt ausschließlich das Instandhaltungsteam durch.

Bei einer Störung meldet die Bedienerin oder der Bediener zunächst an die Schichtleitung. Die Schichtleitung entscheidet über die Priorität und informiert die Instandhaltung. Während der Kernzeit von 6:00 bis 22:00 Uhr ist die Instandhaltung über die Leitstelle erreichbar. Außerhalb der Kernzeit, an Wochenenden und an Feiertagen ist bei Störungen mit hoher Priorität die Rufbereitschaft der Instandhaltung unter der internen Nummer 4711 anzurufen. Bei Störungen mit mittlerer oder niedriger Priorität genügt außerhalb der Kernzeit ein Ticket, das am nächsten Arbeitstag bearbeitet wird.

Kann die Instandhaltung eine Störung nicht innerhalb von vier Stunden beheben, informiert sie die Produktionsleitung und entscheidet gemeinsam mit ihr, ob ein externer Servicepartner beauftragt wird. Die Kontakte der Servicepartner stehen in Kapitel 5.2.

### 1.3 Arbeitssicherheit

Vor jeder Wartung, Reinigung oder Störungsbehebung im Gefahrenbereich einer Maschine ist die Maschine freizuschalten. Das Vorgehen folgt immer denselben fünf Schritten: Maschine stillsetzen, Hauptschalter ausschalten, Hauptschalter mit persönlichem Vorhängeschloss gegen Wiedereinschalten sichern, Spannungsfreiheit und Druckfreiheit prüfen, Arbeitsbereich kennzeichnen. Jede Person, die im Gefahrenbereich arbeitet, hängt ihr eigenes Schloss an. Die Maschine darf erst wieder eingeschaltet werden, wenn alle Schlösser entfernt sind.

Arbeiten an Hydraulikanlagen sind nur im drucklosen Zustand erlaubt. Druckspeicher müssen vorher entladen werden, der Manometerwert muss null zeigen. Arbeiten in der Roboter-Schweißzelle erfolgen grundsätzlich zu zweit, eine Person bleibt außerhalb der Zelle und hat Zugriff auf den Not-Aus. Beim Umgang mit Kühlschmierstoff sind Schutzhandschuhe aus Nitril und eine Schutzbrille zu tragen. Hautschutzplan und Betriebsanweisung hängen an jeder Maschine mit Kühlschmierstoff aus.

Defekte Schutzeinrichtungen wie Türschalter, Lichtgitter oder Not-Aus-Taster dürfen niemals überbrückt werden. Ist eine Schutzeinrichtung defekt, bleibt die Maschine außer Betrieb, bis die Instandhaltung sie instand gesetzt und geprüft hat.

### 1.4 Dokumentation von Störungen und Wartungen

Jede Störung wird als Ticket im Instandhaltungssystem erfasst. Störungen mit hoher Priorität sind innerhalb von 30 Minuten nach Feststellung als Ticket anzulegen, alle anderen bis zum Ende der Schicht. Das Ticket enthält Maschine, Zeitpunkt, Beschreibung, vermutete Ursache, Priorität und Stillstandszeit.

Für die Priorität gelten im ganzen Werk dieselben Definitionen. Hoch bedeutet: Die Maschine steht oder es besteht ein Sicherheitsrisiko. Mittel bedeutet: Die Maschine läuft eingeschränkt, Qualität oder Tempo leiden. Niedrig bedeutet: kein Einfluss auf die laufende Produktion.

Durchgeführte Wartungen werden mit Datum, Tätigkeit, verbrauchten Ersatzteilen und Namen im Wartungsplan der Maschine eingetragen. Bei Abweichungen vom Sollzustand, etwa ungewöhnlichem Verschleiß, ist zusätzlich ein Hinweis-Ticket anzulegen, damit die Intervalle überprüft werden können.

## 2 Betriebsstoffe

### 2.1 Schmierstoffe und Öle

Im Werk werden nur die folgenden freigegebenen Betriebsstoffe verwendet. Andere Produkte dürfen nur nach Freigabe durch die Instandhaltung eingesetzt werden, weil falsche Schmierstoffe Dichtungen angreifen und Lager schädigen können.

| Anwendung | Bezeichnung | Gebinde | Lagerort |
|---|---|---|---|
| Hydraulikaggregate aller Maschinen | Hydrauliköl HLP 46 | 20-Liter-Kanister | Gefahrstofflager, Regal G-2 |
| Spindellager der Drehmaschine | Wälzlagerfett NW-LX2 | 400-Gramm-Kartusche | Ersatzteillager, Regal C-04 |
| Führungsbahnen der Fräszentren | Gleitbahnöl CGLP 68 | 20-Liter-Kanister | Gefahrstofflager, Regal G-3 |
| Zentralschmierung der Montagelinie | Kettenöl NW-K100 | 5-Liter-Kanister | Gefahrstofflager, Regal G-3 |
| Kühlschmierstoff aller Zerspanungsmaschinen | Emulsion NW-Cut 300 (Konzentrat) | 200-Liter-Fass | Gefahrstofflager, Bereich G-1 |

Leere Gebinde und Altöl werden in den gekennzeichneten Sammelbehältern im Gefahrstofflager entsorgt. Ölgetränkte Putzlappen gehören in die roten Sicherheitsbehälter an den Maschinen.

### 2.2 Pflege des Kühlschmierstoffs

Die Fräszentren, die Drehmaschine und die Rundschleifmaschine arbeiten mit der wassergemischten Emulsion NW-Cut 300. Ein gut gepflegter Kühlschmierstoff verlängert die Standzeit der Werkzeuge, verbessert die Oberflächengüte und schützt die Haut der Beschäftigten.

Die Konzentration soll zwischen 6 bis 8 Prozent liegen. Sie wird einmal pro Woche mit dem Refraktometer gemessen, der Messwert wird mit dem Faktor 1,1 multipliziert (Refraktometerfaktor des Konzentrats). Der pH-Wert soll zwischen 8,5 und 9,3 liegen, der Nitritgehalt unter 20 mg/l. Alle drei Werte werden im Kühlschmierstoff-Protokoll an der Maschine eingetragen.

Ist die Konzentration zu niedrig, wird Konzentrat über das Mischgerät nachdosiert, nie unverdünnt in den Tank gegeben. Ist sie zu hoch, wird mit Wasser nachgefüllt. Schäumt der Kühlschmierstoff stark, ist zuerst der Füllstand zu prüfen, denn zu wenig Flüssigkeit im Tank führt zu Luftansaugung an der Pumpe. Hilft das nicht, wird nach Rücksprache mit der Instandhaltung Entschäumer NW-AF 2 in einer Dosierung von 0,05 Prozent zugegeben. Riecht der Kühlschmierstoff faulig oder liegt der pH-Wert unter 8,0, ist er biologisch belastet und muss gewechselt werden. Unabhängig davon wird der Kühlschmierstoff spätestens alle sechs Monate vollständig gewechselt, einschließlich Reinigung des Tanks.

### 2.3 Druckluft

Die zentrale Druckluftversorgung liefert einen Netzdruck von 6 bar. An jeder Maschine sitzt eine Wartungseinheit aus Filter, Druckregler und Wasserabscheider. Der Wasserabscheider wird täglich zu Schichtbeginn entleert, sofern er keinen automatischen Ablass hat. Das Filterelement wird halbjährlich gewechselt oder früher, wenn die Differenzdruckanzeige rot zeigt.

Fällt der Druck an einer Maschine unter 5,5 bar, schaltet die Maschinensteuerung in den meisten Fällen mit einer Warnung ab. Druckschwankungen haben häufig eine von drei Ursachen: ein verschmutztes Filterelement, eine Leckage an Schläuchen oder Kupplungen oder ein zu hoher gleichzeitiger Verbrauch in der Halle. Leckagen lassen sich mit Lecksuchspray finden. Undichte Kupplungen werden getauscht, nicht abgedichtet.

## 3 Maschinen

### 3.1 M-01 CNC-Fräszentrum 1

#### Kurzbeschreibung

Das CNC-Fräszentrum 1 ist ein vertikales Bearbeitungszentrum vom Typ Kessler KFZ 800, Baujahr 2018, Standort Halle 1, Platz 4. Es bearbeitet vor allem Gehäuseteile aus Aluminium. Die Arbeitsspindel erreicht 12.000 Umdrehungen pro Minute. Das Werkzeugmagazin hat 24 Plätze. Verfahrwege: X 800 mm, Y 500 mm, Z 500 mm.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Späne aus dem Arbeitsraum entfernen, Sichtfenster reinigen | täglich | Bedienung |
| Füllstand Gleitbahnöl prüfen | täglich | Bedienung |
| Kühlschmierstoff messen (Kapitel 2.2) | wöchentlich | Bedienung |
| Abstreifer der Führungsbahnen prüfen | monatlich | Instandhaltung |
| Spindelrundlauf mit Prüfdorn messen, Grenzwert 5 µm | alle 1.000 Betriebsstunden | Instandhaltung |
| Hydrauliköl wechseln, Filter tauschen | jährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Achsantrieb meldet Überstrom | Schwergängige Führung, Späne unter dem Abstreifer, Motor defekt | Maschine freischalten, Abstreifer reinigen; tritt der Fehler erneut auf, Instandhaltung rufen |
| Kühlmittel tropft aus Leitung | Schlauchschelle locker, Schlauch porös | Auffangwanne unterstellen, Ticket mit Priorität niedrig, Schlauch beim nächsten Stillstand tauschen |
| CNC-Programm lässt sich nicht laden | Programm fehlerhaft übertragen, Speicher voll | Programm erneut aus der Programmverwaltung laden, alte Programme archivieren |

#### Wichtige Ersatzteile

| Teil | Teilenummer | Lagerort |
|---|---|---|
| Abstreifer Führungsbahn X | KFZ-8810 | Regal A-03 |
| Kühlmittelschlauch 12 mm, Meterware | NW-SL12 | Regal A-07 |

### 3.2 M-02 CNC-Fräszentrum 2

#### Kurzbeschreibung

Das CNC-Fräszentrum 2 ist baugleich zum Fräszentrum 1, jedoch vom neueren Typ Kessler KFZ 800 plus, Baujahr 2021, Standort Halle 1, Platz 5. Unterschiede: Das Werkzeugmagazin hat 40 Plätze statt 24, und die Maschine besitzt einen Messtaster zur automatischen Werkstückvermessung. Es fertigt vorwiegend Teile aus Stahl.

#### Wartungsintervalle

Es gelten dieselben Intervalle wie bei M-01. Zusätzlich gilt:

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Greifer des Werkzeugwechslers reinigen und auf Verschleiß prüfen | monatlich | Instandhaltung |
| Messtaster kalibrieren | vierteljährlich | Instandhaltung |
| Kugelgewindetriebe auf Spiel prüfen | alle 2.000 Betriebsstunden | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Werkzeugwechsler hakt, Wechsel dauert deutlich länger | Verschmutzte Greiferfinger, verschlissene Federn | Greifer reinigen; bei Verschleiß Federn tauschen (Teilenummer KFZ-9120) |
| Quietschen an einer Achse | Schmierung der Führung unzureichend | Zentralschmierung manuell auslösen, Gleitbahnöl-Füllstand prüfen; Geräusch weiter beobachten |
| Messtaster liefert stark streuende Werte | Tastkugel verschmutzt, Kalibrierung veraltet | Tastkugel reinigen, Messtaster neu kalibrieren |

### 3.3 M-03 CNC-Drehmaschine

#### Kurzbeschreibung

Die CNC-Drehmaschine ist eine Schrägbettmaschine vom Typ Hartwig TD 400, Baujahr 2015, Standort Halle 1, Platz 9. Sie fertigt Wellen und Bolzen bis 400 mm Länge. Das Hydraulikaggregat versorgt das Spannfutter und den Reitstock. Wegen ihres Alters ist sie die störanfälligste Maschine in Halle 1.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Späne entfernen, Spannfutter abblasen | täglich | Bedienung |
| Ölstand im Hydraulikaggregat prüfen (Hydrauliköl HLP 46) | täglich | Bedienung |
| Spindellager mit Wälzlagerfett NW-LX2 nachschmieren, Menge 8 Gramm je Lager | alle 500 Betriebsstunden | Instandhaltung |
| Spannfutter zerlegen, reinigen, Backen fetten | alle 250 Betriebsstunden | Instandhaltung |
| Spindelrundlauf messen, Grenzwert 8 µm | alle 1.000 Betriebsstunden | Instandhaltung |
| Hydrauliköl wechseln, Rücklauffilter tauschen | jährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Spindel vibriert, Teile außerhalb der Toleranz | Spindellager verschlissen oder unzureichend geschmiert | Drehzahl reduzieren, Instandhaltung informieren; Rundlauf messen, Lager nachschmieren oder tauschen |
| Maschine schaltet mit Meldung „Hydraulikdruck“ ab | Ölstand unter Minimum, Leckage | Ölstand prüfen; nur mit HLP 46 auffüllen; Leckage suchen, Ticket mit Priorität hoch |
| Spannfutter spannt ungleichmäßig | Späne in der Backenführung | Spannfutter reinigen; bei Wiederholung zerlegen |

#### Wichtige Ersatzteile

| Teil | Teilenummer | Lagerort |
|---|---|---|
| Ersatz-Spannfutter dreibackig, 250 mm | HTD-2250 | Regal C-12 |
| Spindellagersatz | HTD-1100 | Regal C-11 |
| Rücklauffilter Hydraulik | HTD-3310 | Regal C-04 |

### 3.4 M-04 Rundschleifmaschine

#### Kurzbeschreibung

Die Rundschleifmaschine vom Typ Brenner RS 600, Baujahr 2019, Standort Halle 2, Platz 2, schleift gehärtete Wellen auf das Endmaß. Die Schleifscheibe hat einen Durchmesser von 500 mm. Der Kühlschmierstoff wird über ein Papierbandfilter gereinigt.

#### Sicherheitshinweise zur Schleifscheibe

Schleifscheiben sind bruchgefährdet. Eine neue Schleifscheibe wird vor der Montage durch eine Klangprobe geprüft: Die Scheibe wird frei hängend mit einem Holzhammer leicht angeschlagen. Ein heller, klarer Klang zeigt eine unbeschädigte Scheibe, ein dumpfer oder scheppernder Klang deutet auf Risse hin. Scheiben mit Rissen oder unklarer Klangprobe dürfen nicht montiert werden. Die zulässige Umfangsgeschwindigkeit der eingesetzten Scheiben beträgt höchstens 50 m/s. Nach jeder Montage wird die Scheibe ausgewuchtet und abgerichtet, anschließend läuft sie mindestens fünf Minuten mit Betriebsdrehzahl, während sich niemand in der Drehebene aufhält.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Schleifscheibe abrichten | je nach Bedarf, spätestens alle 50 Werkstücke | Bedienung |
| Papierbandfilter prüfen, Rolle wechseln | wenn die Füllstandsanzeige gelb zeigt | Bedienung |
| Kühlschmierstoff messen (Kapitel 2.2) | wöchentlich | Bedienung |
| Wuchteinrichtung prüfen | monatlich | Instandhaltung |
| Schleifspindel auf Lagergeräusche prüfen | vierteljährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Risse an der Schleifscheibe | Schlag, Überlastung, falsche Lagerung | Maschine sofort stoppen und freischalten, Scheibe tauschen, Ticket mit Priorität hoch |
| Rattermarken auf dem Werkstück | Scheibe unwuchtig oder stumpf | Scheibe auswuchten und abrichten |
| Englische Bedienoberfläche nach Update | Spracheinstellung zurückgesetzt | Im Menü Einstellungen, Sprache auf Deutsch stellen; kein Einfluss auf die Produktion |

### 3.5 M-05 Roboter-Schweißzelle

#### Kurzbeschreibung

Die Roboter-Schweißzelle besteht aus einem Sechsachs-Roboter, der Schweißstromquelle Varion 500, einem Drehtisch mit zwei hydraulischen Spannvorrichtungen und einem Lichtgitter an der Beladeseite. Baujahr 2020, Standort Halle 2, Platz 6. Geschweißt wird im MAG-Verfahren mit Mischgas.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Brenner reinigen, Gasdüse und Kontaktrohr prüfen | alle vier Stunden Schweißbetrieb | Bedienung |
| Lichtgitter mit Prüfstab testen | täglich zu Schichtbeginn | Bedienung |
| Schutzgasflasche wechseln | wenn der Flaschendruck unter 10 bar fällt | Bedienung |
| Kühlflüssigkeit des Brenners prüfen | wöchentlich | Bedienung |
| Hydraulik der Spannvorrichtungen auf Leckagen prüfen | monatlich | Instandhaltung |
| Roboter-Getriebeöl wechseln | alle 20.000 Betriebsstunden | Servicepartner |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Schweißstromquelle zeigt Fehlercode | siehe Kapitel 5.1 | Fehlercode ablesen und nach Tabelle vorgehen |
| Lichtgitter löst aus, obwohl sich niemand im Schutzfeld befindet | Verschmutzte Linsen, Schweißspritzer auf Sender oder Empfänger, verstellte Ausrichtung | Linsen mit weichem Tuch reinigen, Ausrichtung prüfen; Lichtgitter niemals überbrücken; bei Wiederholung Instandhaltung rufen |
| Spannvorrichtung öffnet sich während des Schweißens | Hydraulikdruck fällt ab, Leckage am Spannzylinder | Zelle sofort stillsetzen, Ticket mit Priorität hoch, Verletzungsgefahr |
| Bildschirm am Bedienpult flackert | Wackelkontakt im Displaykabel | Ticket mit Priorität niedrig, Bedienung bleibt möglich |

### 3.6 M-06 Montagelinie A

#### Kurzbeschreibung

Die Montagelinie A verbindet sechs Montagestationen über ein Werkstückträger-Fördersystem. Gesteuert wird sie von einer zentralen SPS, die über das Werksnetz mit dem Leitrechner der Produktion verbunden ist. Lichtschranken an der Zuführung erkennen die Werkstückträger. Baujahr 2017, Umbau 2023, Standort Halle 2, Bereich M.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Lichtschranken an Zuführung und Stationen reinigen | täglich | Bedienung |
| Not-Aus-Taster aller Stationen auf Funktion prüfen | wöchentlich | Schichtleitung |
| Kettenspannung des Fördersystems prüfen | monatlich | Instandhaltung |
| Zentralschmierung auffüllen (Kettenöl NW-K100) | wenn die Anzeige rot zeigt | Instandhaltung |
| Datensicherung der SPS | nach jeder Programmänderung, mindestens vierteljährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Not-Aus lässt sich nicht zurücksetzen | Taster mechanisch verklemmt, Fehler im Sicherheitskreis | Alle Not-Aus-Taster der Linie prüfen; bleibt der Fehler, Instandhaltung rufen; Linie steht, Priorität hoch |
| SPS verliert die Verbindung zum Leitrechner | Netzwerkkabel, Switch, IP-Konflikt | Linkanzeige am Switch prüfen; Neustart der SPS nur durch die Instandhaltung |
| Kurze Unterbrechungen durch Lichtschranke | Staub auf der Optik | Lichtschranke reinigen, bei Wiederholung Ausrichtung prüfen |

### 3.7 M-07 Prüfstand

#### Kurzbeschreibung

Der Prüfstand prüft fertig montierte Baugruppen auf Dichtheit und Funktion. Er besteht aus einer Druckprüfeinheit mit Druckluft, einem Messrechner mit Prüfsoftware und einem Etikettendrucker. Baujahr 2022, Standort Halle 2, Bereich P. Die Messergebnisse werden automatisch auf dem Qualitätsserver gespeichert.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Prüfadapter und Dichtungen auf Beschädigung prüfen | täglich | Bedienung |
| Leckprüfung mit Referenzteil | zu jedem Schichtbeginn | Bedienung |
| Kalibrierung der Drucksensoren | jährlich durch einen akkreditierten Dienstleister | Qualitätssicherung |
| Elektrische Prüfung nach DGUV Vorschrift 3 | jährlich | Elektrofachkraft |
| Datensicherung des Messrechners prüfen | monatlich | IT |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Messprotokolle werden nicht gespeichert | Verbindung zum Qualitätsserver gestört, Speicher voll | Werte von Hand im Ersatzprotokoll notieren, IT informieren, Priorität mittel |
| Messungen streuen, Referenzteil fällt durch | Druckluft schwankt, Dichtung am Adapter verschlissen | Netzdruck an der Wartungseinheit prüfen (Kapitel 2.3), Dichtung tauschen |
| Prüfplakette abgelaufen | Prüftermin verpasst | Elektrofachkraft informieren, Prüfung nachholen |

### 3.8 M-08 Teilewaschanlage

#### Kurzbeschreibung

Die Teilewaschanlage reinigt Teile nach der Zerspanung von Spänen und Kühlschmierstoff. Sie arbeitet mit wässriger, alkalischer Waschlauge bei 60 °C, die über einen elektrischen Heizstab erwärmt wird. Ein Förderband transportiert die Teilekörbe durch Wasch-, Spül- und Trockenzone. Baujahr 2016, Standort Halle 1, Platz 12.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Siebe in der Waschzone reinigen | täglich | Bedienung |
| Temperatur und Füllstand der Waschlauge prüfen | täglich | Bedienung |
| Ölabscheider leeren | wöchentlich | Bedienung |
| Waschbad komplett wechseln, Tank reinigen | alle vier Wochen | Instandhaltung |
| Förderband auf Spannung und Verschleiß prüfen | monatlich | Instandhaltung |
| Heizstab entkalken | halbjährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Teile bleiben fettig | Waschlauge zu kalt oder verbraucht | Temperatur prüfen; liegt sie deutlich unter 60 °C, Heizstab und Sicherung prüfen lassen; Bad wechseln, wenn es älter als vier Wochen ist |
| Förderband steht | Band gerissen oder Antrieb blockiert | Anlage freischalten, Ursache prüfen, Priorität hoch |
| Waschzyklen zu kurz nach Stromausfall | Zeitsteuerung auf Werkseinstellung zurückgesetzt | Parameter aus der Parameterliste im Schaltschrank neu eingeben |

## 4 Ersatzteillager und Bestellung

Das Ersatzteillager befindet sich in Halle 2 neben der Werkstatt der Instandhaltung. Die Regale sind nach Maschinen gruppiert: Regalreihe A für die Fräszentren, B für die Schleifmaschine, C für die Drehmaschine, D für die Schweißzelle, E für Montagelinie und Prüfstand, F für die Waschanlage. Gefahrstoffe lagern getrennt im Gefahrstofflager.

Für kritische Teile gelten Mindestbestände. Wird ein Teil entnommen, trägt die entnehmende Person die Entnahme in die Lagerliste ein. Fällt der Bestand unter den Mindestbestand, bestellt die Instandhaltung über den Einkauf nach. Bestellungen bis 500 Euro gibt die Leitung Instandhaltung frei, darüber die Werkleitung. Für Eilbestellungen bei Stillstand mit hoher Priorität darf die Rufbereitschaft auch außerhalb der Kernzeit direkt beim Servicepartner bestellen und dokumentiert dies am nächsten Arbeitstag.

| Teil | Mindestbestand | Übliche Lieferzeit |
|---|---|---|
| Schleifscheibe 500 mm für M-04 | 2 Stück | 3 Wochen |
| Spindellagersatz HTD-1100 für M-03 | 1 Satz | 6 Wochen |
| Kontaktrohre für den Schweißbrenner | 50 Stück | 1 Woche |
| Lichtschranke Zuführung M-06 | 1 Stück | 2 Wochen |

## 5 Anhang

### 5.1 Fehlercodes der Schweißstromquelle Varion 500

| Code | Bedeutung | Maßnahme |
|---|---|---|
| E10 | Netzspannung außerhalb des zulässigen Bereichs | Netzversorgung prüfen lassen |
| E21 | Übertemperatur im Kühlkreislauf des Brenners | Kühlflüssigkeit prüfen und auffüllen, Kühler auf Verschmutzung prüfen; Anlage abkühlen lassen |
| E33 | Drahtvorschub blockiert | Drahtführung und Vorschubrollen prüfen, Drahtspule auf Verwicklungen prüfen |
| E41 | Kein Gasdurchfluss | Flaschenventil und Druckminderer prüfen, Flasche bei unter 10 bar wechseln |
| E45 | Kommunikation zur Robotersteuerung unterbrochen | Steckverbindung des Buskabels prüfen, Instandhaltung rufen |

### 5.2 Kontakte

| Bereich | Kontakt |
|---|---|
| Leitstelle Instandhaltung (Kernzeit 6 bis 22 Uhr) | intern 4700 |
| Rufbereitschaft Instandhaltung (außerhalb der Kernzeit) | intern 4711 |
| IT-Service | intern 4800 |
| Servicepartner Fräszentren (Kessler Service) | Servicevertrag SV-1180, Hotline im Intranet |
| Servicepartner Schweißzelle | Servicevertrag SV-2045, Hotline im Intranet |
| Kalibrierdienstleister Prüfstand | Auftrag über die Qualitätssicherung |
'''

def _maschine(machine_id):
    for m in MASCHINEN:
        if m["id"] == machine_id:
            return m
    raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig: " + ", ".join(m["id"] for m in MASCHINEN))

def get_machine_status(machine_id):
    m = _maschine(machine_id)
    offen = [t["ticket"] + ": " + t["beschreibung"] for t in TICKETS if t["maschine"] == machine_id and t["status"] == "offen"]
    return {"id": m["id"], "name": m["name"], "status": m["status"], "betriebsstunden": m["betriebsstunden"], "letzte_wartung": m["letzte_wartung"], "offene_tickets": offen}

def get_maintenance_history(machine_id, limit=5):
    _maschine(machine_id)
    e = sorted((w for w in WARTUNGSHISTORIE if w[0] == machine_id), key=lambda w: w[1], reverse=True)
    return [{"datum": d, "taetigkeit": t, "durchgefuehrt_von": v} for _, d, t, v in e[:limit]]

def get_sensor_data(machine_id):
    _maschine(machine_id)
    bew = lambda w, g, a: "ÜBERSCHRITTEN" if (a == "max" and w > g) or (a == "min" and w < g) else "ok"
    return [{"messgroesse": n, "wert": w, "einheit": e, "grenzwert": f"{'max.' if a == 'max' else 'min.'} {g}", "bewertung": bew(w, g, a)} for n, w, e, g, a in SENSORDATEN[machine_id]]

def chunk_struktur(markdown):
    chunks, pfad, zeilen, titel = [], {}, [], ""
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = " › ".join(pfad[e] for e in sorted(pfad) if e > 1) or pfad[ebene]
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = chunk_struktur(HANDBUCH)
_tok = lambda t: re.findall(r"[a-zäöüß0-9-]+", t.lower())
_bm25 = BM25Okapi([_tok(c["titel"] + " " + c["text"]) for c in CHUNKS])
def search_knowledge_base(query, k=3):
    s = _bm25.get_scores(_tok(query))
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in sorted(range(len(CHUNKS)), key=lambda i: s[i], reverse=True)[:k]]

ID = {"type": "string", "description": "Maschinen-ID im Format M-01 bis M-08."}
TOOLS_BASIS = [
    {"name": "get_machine_status", "description": "Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, offene Tickets.",
     "parameters": {"type": "object", "properties": {"machine_id": ID}, "required": ["machine_id"]}},
    {"name": "get_maintenance_history", "description": "Letzte durchgeführte Wartungen einer Maschine, neueste zuerst.",
     "parameters": {"type": "object", "properties": {"machine_id": ID, "limit": {"type": "integer"}}, "required": ["machine_id"]}},
    {"name": "get_sensor_data", "description": "Aktuelle Messwerte einer Maschine mit Grenzwert und Bewertung.",
     "parameters": {"type": "object", "properties": {"machine_id": ID}, "required": ["machine_id"]}},
    {"name": "search_knowledge_base", "description": "Durchsucht das Wartungshandbuch: Intervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
]
FUNKTIONEN = {"get_machine_status": get_machine_status, "get_maintenance_history": get_maintenance_history,
              "get_sensor_data": get_sensor_data, "search_knowledge_base": search_knowledge_base}

SYSTEM = f"""Du bist der Instandhaltungs-Agent der Nordwerk GmbH. Heute ist der {HEUTE:%d.%m.%Y}.
Nutze die Tools, statt Fakten zu raten. Antworte knapp auf Deutsch, in höchstens fünf Sätzen."""

def fuehre_tool_aus(name, eingabe):
    try:
        if name not in FUNKTIONEN:
            raise ValueError(f"Unbekanntes Tool '{name}'.")
        return FUNKTIONEN[name](**eingabe), False
    except Exception as e:
        return {"fehler": str(e)}, True

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def argumente(tc):
    """Die Argumente eines Tool-Aufrufs kommen als JSON-String."""
    try:
        return json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"_ungueltig": tc.function.arguments}

def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    inhalt = json.dumps(ergebnis, ensure_ascii=False)
    return {"role": "tool", "tool_call_id": tool_call_id, "content": ("FEHLER: " + inhalt) if is_error else inhalt}

def gecachte_tokens(u):
    """Manche Server melden, wie viele Input-Tokens aus einem Cache kamen. Fehlt die Angabe, ist es 0."""
    details = getattr(u, "prompt_tokens_details", None)
    return (getattr(details, "cached_tokens", 0) or 0) if details else 0

class Gespraech:
    """Ein Gespräch über mehrere Fragen. kompaktierer: Funktion, die den Verlauf kürzt (Teil B)."""
    def __init__(self, system=SYSTEM, tools=None, kompaktierer=None, schwelle=None):
        self.system, self.tools = system, tools or TOOLS_BASIS
        self.kompaktierer, self.schwelle = kompaktierer, schwelle
        self.nachrichten, self.protokoll = [], []   # Verlauf ohne System-Nachricht, die kommt bei jedem Aufruf vorne dazu

    def frage(self, text, max_runden=8):
        if self.kompaktierer and self.schwelle and self.protokoll and self.protokoll[-1]["kontext"] > self.schwelle:
            self.nachrichten = self.kompaktierer(self.nachrichten)
            print("   🗜️  Verlauf kompaktiert")
        self.nachrichten.append({"role": "user", "content": text})
        summe = {"input": 0, "cache_write": 0, "cache_read": 0, "output": 0}
        kontext, antwort = 0, None
        for _ in range(max_runden):
            antwort = client.chat.completions.create(model=MODEL, max_tokens=800, tools=als_tools(self.tools),
                                                     messages=[{"role": "system", "content": self.system}] + self.nachrichten)
            u = antwort.usage
            summe["cache_read"] += gecachte_tokens(u)
            summe["input"] += u.prompt_tokens - gecachte_tokens(u); summe["output"] += u.completion_tokens
            kontext = u.prompt_tokens
            msg = antwort.choices[0].message
            self.nachrichten.append(assistent_nachricht(msg))
            if not msg.tool_calls:
                break
            for tc in msg.tool_calls:
                erg, fehler = fuehre_tool_aus(tc.function.name, argumente(tc))
                self.nachrichten.append(tool_nachricht(tc.id, erg, fehler))
        self.protokoll.append({"frage": text, **summe, "kontext": kontext})
        return antwort.choices[0].message.content or ""

SCHICHT = [
    "Guten Morgen. Wie ist der Status von M-07?",
    "Und die Sensordaten dazu?",
    "Was sagt das Handbuch zur elektrischen Prüfung?",
    "Wann war die letzte elektrische Prüfung an M-07?",
    "Okay. Wie sieht es an der Drehmaschine aus?",
    "Gibt es offene Tickets an M-03?",
    "Was bedeutet E21 an der Schweißzelle?",
    "Fass bitte zusammen: Was hatten wir heute zu M-07 festgestellt, und was ist zu tun?",
]
print("Bereit:", len(SCHICHT), "Fragen in der Schicht")

---
# Teil A · Kontext als Budget

## A1 · Woraus besteht der Kontext?

Jede Anfrage enthält System-Prompt, Tool-Definitionen und den Verlauf. Mit Mini-Anfragen (`max_tokens=1`) und `usage.prompt_tokens` lässt sich messen, wie viel jeder Teil belegt (Woche 4).

### TODO 1 · Bilanz
Ergänzt `kontext_bilanz`. Zählt dreimal mit derselben Platzhalter-Nachricht `[{"role": "user", "content": "x"}]`: ohne System und Tools, nur mit System, mit System und Tools. Zählt dann ein viertes Mal mit System, Tools und dem echten Verlauf. Die Differenzen ergeben die Anteile.

In [ ]:
def zaehle(system=None, tools=None, nachrichten=None):
    """Zählt die Input-Tokens über eine Mini-Anfrage (max_tokens=1)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + (nachrichten or [{"role": "user", "content": "x"}])
    args = {"model": MODEL, "max_tokens": 1, "messages": msgs}
    if tools: args["tools"] = als_tools(tools)
    return client.chat.completions.create(**args).usage.prompt_tokens

def kontext_bilanz(system, tools, nachrichten):
    # TODO 1: gebt {"system": ..., "tools": ..., "verlauf": ..., "gesamt": ...} zurück
    pass

## A2 · Eine Schicht ohne Gegenmaßnahmen

Acht Fragen, wie sie ein Instandhalter im Lauf einer Schicht stellen könnte. Beobachtet, wie der Kontext pro Frage wächst.

In [ ]:
g_basis = Gespraech()
for f in SCHICHT:
    print(f"❓ {f}\n💬 {g_basis.frage(f)[:220]}\n")

bilanz = kontext_bilanz(SYSTEM, TOOLS_BASIS, g_basis.nachrichten)
check("TODO 1 Bilanz", lambda: set(bilanz) == {"system", "tools", "verlauf", "gesamt"} and bilanz["verlauf"] > bilanz["system"] > 0)
if bilanz:
    print("Kontext am Ende der Schicht:", bilanz)

### TODO 2 · Was würde das kommerziell kosten?
In der KI-Toolbox zahlt ihr nichts. Für die Einordnung rechnen wir trotzdem mit kommerziellen Referenzpreisen. Viele Anbieter rechnen Tokens, die aus einem Cache kommen, deutlich billiger ab (Prompt Caching, Teil D).
`kosten_cent(z)` bekommt ein Dictionary mit den Token-Zahlen `input`, `cache_write`, `cache_read` und `output` und gibt die Kosten in Cent zurück. Nutzt `PREISE` (Dollar pro 1 Mio. Tokens).

In [ ]:
def kosten_cent(z):
    # TODO 2
    pass

In [ ]:
check("TODO 2a nur Input", lambda: math.isclose(kosten_cent({"input": 10_000, "cache_write": 0, "cache_read": 0, "output": 0}), 1.0))
check("TODO 2b Cache-Lesen ist ein Zehntel", lambda: math.isclose(kosten_cent({"input": 0, "cache_write": 0, "cache_read": 10_000, "output": 0}), 0.1))
check("TODO 2c alle Arten", lambda: math.isclose(kosten_cent({"input": 1000, "cache_write": 4000, "cache_read": 20000, "output": 500}), (1000 + 5000 + 2000 + 2500) / 1e4))

def zeige_verlauf(gespraeche):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for name, g in gespraeche.items():
        ax.plot(range(1, len(g.protokoll) + 1), [p["kontext"] for p in g.protokoll], marker="o", label=name)
    ax.set_xlabel("Frage"); ax.set_ylabel("Kontext der letzten Anfrage (Tokens)"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()
    for name, g in gespraeche.items():
        print(f"{name:28s} kommerziell {sum(kosten_cent(p) for p in g.protokoll):.2f} Cent | Tokens: "
              + ", ".join(f"{a} {sum(p[a] for p in g.protokoll):,}".replace(",", ".") for a in ["input", "cache_write", "cache_read", "output"]))

if kosten_cent({"input": 1, "cache_write": 0, "cache_read": 0, "output": 0}) is not None:
    zeige_verlauf({"ohne Gegenmaßnahmen": g_basis})

> **Zum Nachdenken:** Welcher Teil des Kontexts wächst? Welche Teile des Verlaufs braucht der Agent bei Frage 8 noch wirklich? Die vollständigen Tool-Ausgaben von Frage 2?

---
# Teil B · Kompaktieren

Ist der Kontext größer als eine Schwelle, ersetzen wir die älteren Runden durch eine **Zusammenfassung** und behalten nur die letzten Fragen vollständig. Geschnitten wird nur **zwischen** zwei Fragen, damit kein Tool-Aufruf ohne sein Tool-Ergebnis übrig bleibt.

### TODO 3 · Wo schneiden?
Eine neue Frage erkennt ihr an einer Nachricht mit der Rolle `user` (Tool-Ergebnisse haben die Rolle `tool`). `finde_schnitt(nachrichten, behalte_letzte)` gibt den Index der `behalte_letzte`-letzten Frage zurück, oder 0, wenn es weniger Fragen gibt.

In [ ]:
def finde_schnitt(nachrichten, behalte_letzte=2):
    # TODO 3
    pass

In [ ]:
_v = [{"role": "user", "content": "F1"}, {"role": "assistant", "content": "", "tool_calls": [{"id": "t1", "type": "function", "function": {"name": "get_machine_status", "arguments": "{}"}}]},
      {"role": "tool", "tool_call_id": "t1", "content": "{}"},
      {"role": "assistant", "content": "A1"}, {"role": "user", "content": "F2"}, {"role": "assistant", "content": "A2"},
      {"role": "user", "content": "F3"}, {"role": "assistant", "content": "A3"}]
check("TODO 3a letzte zwei Fragen behalten → Schnitt bei Index 4", lambda: finde_schnitt(_v, 2) == 4)
check("TODO 3b Tool-Ergebnisse zählen nicht als Frage", lambda: finde_schnitt(_v, 3) == 0 and finde_schnitt(_v, 1) == 6)

### TODO 4 · Kompaktieren
`kompaktiere(nachrichten, behalte_letzte=2)`:
1. Schnitt mit `finde_schnitt` bestimmen. Ist er 0, die Nachrichten unverändert zurückgeben.
2. Den alten Teil mit `als_text` (vorgegeben) in Text umwandeln und mit `SYSTEM_ZUSAMMENFASSUNG` als System-Nachricht zusammenfassen lassen.
3. Neue Liste: eine `user`-Nachricht `"Zusammenfassung des bisherigen Gesprächs:\n" + zusammenfassung`, eine `assistant`-Nachricht `"Verstanden."`, danach der behaltene Teil.

Schreibt auch den Zusammenfassungs-Prompt: Was **muss** erhalten bleiben, was darf weg?

In [ ]:
def als_text(nachrichten):
    zeilen = []
    for n in nachrichten:
        if n["role"] == "tool":
            zeilen.append(f"Tool-Ergebnis: {n['content'][:600]}")
            continue
        if n.get("content"):
            zeilen.append(f"{n['role']}: {n['content']}")
        for tc in n.get("tool_calls", []):
            zeilen.append(f"Tool-Aufruf: {tc['function']['name']}({tc['function']['arguments']})")
    return "\n".join(zeilen)

# TODO 4a
SYSTEM_ZUSAMMENFASSUNG = """

"""

def kompaktiere(nachrichten, behalte_letzte=2):
    # TODO 4b
    pass

In [ ]:
_k = kompaktiere(copy.deepcopy(_v), 2) if kompaktiere(copy.deepcopy(_v[:2]), 2) is not None else None
check("TODO 4a Zusammenfassungs-Prompt sagt, was bleiben muss", lambda: len(SYSTEM_ZUSAMMENFASSUNG.strip()) > 80)
check("TODO 4b neue Struktur", lambda: len(_k) == 6 and _k[0]["content"].startswith("Zusammenfassung") and _k[1]["role"] == "assistant" and _k[2]["content"] == "F2")
check("TODO 4c kurzer Verlauf bleibt unverändert", lambda: kompaktiere(_v[:2], 2) == _v[:2])

In [ ]:
if _k:
    g_kompakt = Gespraech(kompaktierer=kompaktiere, schwelle=6000)
    for f in SCHICHT:
        antwort = g_kompakt.frage(f)
    print("Letzte Antwort mit Kompaktierung:\n💬", antwort)
    zeige_verlauf({"ohne Gegenmaßnahmen": g_basis, "mit Kompaktierung": g_kompakt})

> **Zum Nachdenken:** Wurde die Zusammenfassungsfrage am Ende trotz Kompaktierung richtig beantwortet? Was ist bei der Kompaktierung verloren gegangen? Und was kostet der zusätzliche Zusammenfassungs-Aufruf?

---
# Teil C · Langzeit-Memory

Kompaktierung hilft **innerhalb** eines Gesprächs. Am nächsten Tag ist der Verlauf weg. Ein **Langzeit-Memory** speichert ausgewählte Notizen dauerhaft, hier in einer JSON-Datei. Der Agent bekommt zwei Tools: `remember` zum Speichern und `recall` zum Abrufen.

### TODO 5 · `remember` und `recall`
- `remember(thema, inhalt)` hängt einen Eintrag `{"id", "zeit", "thema", "inhalt"}` an die Liste in `MEMORY_DATEI` an (Datei anlegen, falls sie fehlt). Die `id` ist fortlaufend. Rückgabe: `{"gespeichert": id}`.
- `recall(suchbegriff)` gibt die bis zu 5 neuesten Einträge zurück, bei denen der Suchbegriff (ohne Groß- und Kleinschreibung) im Thema oder Inhalt vorkommt.

In [ ]:
MEMORY_DATEI = "memory.json"

def lade_memory():
    if not os.path.exists(MEMORY_DATEI):
        return []
    with open(MEMORY_DATEI, encoding="utf-8") as f:
        return json.load(f)

def speichere_memory(eintraege):
    with open(MEMORY_DATEI, "w", encoding="utf-8") as f:
        json.dump(eintraege, f, ensure_ascii=False, indent=1)

def remember(thema, inhalt):
    # TODO 5a
    pass

def recall(suchbegriff):
    # TODO 5b
    pass

In [ ]:
if os.path.exists(MEMORY_DATEI): os.remove(MEMORY_DATEI)
check("TODO 5a speichern", lambda: remember("M-07", "Lüfter defekt")["gespeichert"] == 1 and remember("M-03", "Vibration erhöht")["gespeichert"] == 2)
check("TODO 5b finden, unabhängig von Groß- und Kleinschreibung", lambda: [e["id"] for e in recall("m-07")] == [1] and len(recall("vibration")) == 1)
check("TODO 5c neueste zuerst", lambda: (remember("M-07", "Elektrofachkraft informiert"), [e["id"] for e in recall("M-07")])[1] == [3, 1])
if os.path.exists(MEMORY_DATEI): os.remove(MEMORY_DATEI)

### TODO 6 · Die Tool-Definitionen
Schreibt die Definitionen für `remember` (Parameter `thema`, `inhalt`) und `recall` (Parameter `suchbegriff`). Die Beschreibung von `remember` muss klar sagen, **was** gespeichert werden darf: fachliche Befunde, Entscheidungen und offene Punkte zu Maschinen. Und **was nicht**: keine personenbezogenen Daten wie Namen, Gesundheit oder Leistungsbewertungen.

In [ ]:
# TODO 6
TOOL_REMEMBER = {}
TOOL_RECALL = {}

In [ ]:
check("TODO 6a beide Tools vollständig", lambda: TOOL_REMEMBER["name"] == "remember" and TOOL_RECALL["name"] == "recall" and TOOL_RECALL["parameters"]["required"] == ["suchbegriff"])
check("TODO 6b Datenschutz-Regel in der Beschreibung", lambda: "personenbezogen" in TOOL_REMEMBER["description"].lower())

### TODO 7 · Datenschutz im Code
Eine Beschreibung ist eine Bitte, keine Garantie (Woche 8). Ergänzt `remember_sicher`: Enthält Thema oder Inhalt einen Namen aus `PERSONEN` **als ganzes Wort** oder ein Wort, das mit einem Begriff aus `SENSIBEL` **beginnt**, wird **nichts** gespeichert, sondern ein `ValueError` mit einer Erklärung ausgelöst.
Tipp: `re.search(rf"\b{re.escape(p)}\b", text, re.IGNORECASE)` findet ganze Wörter. Ein einfaches `p in text` würde etwa „Wolf“ auch in „Wolfram“ finden. Schreibt außerdem `vergiss(id)`, das einen Eintrag löscht, denn wer speichert, muss auch löschen können.

Diskutiert: Was rutscht durch diesen einfachen Filter trotzdem durch?

In [ ]:
PERSONEN = ["Yilmaz", "Becker", "Nguyen", "Schäfer", "Wolf", "Kowalski"]     # Schichtleitungen aus dem Schichtbuch
SENSIBEL = ["krank", "arzt", "privat", "leistungsbewertung", "beurteilung", "abmahnung", "fehlzeit"]

def remember_sicher(thema, inhalt):
    # TODO 7a
    pass

def vergiss(id):
    # TODO 7b: Eintrag mit dieser id entfernen, {"geloescht": id} zurückgeben, bei unbekannter id ValueError
    pass

In [ ]:
if os.path.exists(MEMORY_DATEI): os.remove(MEMORY_DATEI)
def _abgelehnt(f, *a):
    try:
        f(*a); return False
    except ValueError:
        return True
check("TODO 7a fachliche Notiz wird gespeichert", lambda: remember_sicher("M-07", "Schaltschranklüfter ausgefallen")["gespeichert"] == 1)
check("TODO 7b Name wird abgelehnt", lambda: _abgelehnt(remember_sicher, "Schicht", "K. Yilmaz hat das Ticket angelegt"))
check("TODO 7c sensible Angabe wird abgelehnt", lambda: _abgelehnt(remember_sicher, "Schicht", "Schichtleitung heute krank"))
check("TODO 7e kein Fehlalarm bei Fachbegriffen", lambda: remember_sicher("M-05", "Wolfram-Elektrode und Leistungsaufnahme geprüft")["gespeichert"] == 2)
check("TODO 7d löschen", lambda: vergiss(1) == {"geloescht": 1} and [e["id"] for e in lade_memory()] == [2] and _abgelehnt(vergiss, 99))

**Zwei Gespräche, ein Gedächtnis.** Im ersten Gespräch soll sich der Agent einen Befund merken. Im zweiten, einem neuen Gespräch ohne gemeinsamen Verlauf, fragt der Nutzer danach.

In [ ]:
if os.path.exists(MEMORY_DATEI): os.remove(MEMORY_DATEI)
if TOOL_REMEMBER and remember_sicher("Test", "x") is not None:
    os.remove(MEMORY_DATEI)
    FUNKTIONEN.update({"remember": remember_sicher, "recall": recall})
    TOOLS_MEMORY = TOOLS_BASIS + [TOOL_REMEMBER, TOOL_RECALL]
    SYSTEM_MEMORY = SYSTEM + "\nDu hast ein Langzeitgedächtnis (remember, recall). Prüfe bei Fragen zu einer Maschine zuerst mit recall, ob es frühere Notizen gibt."

    tag1 = Gespraech(system=SYSTEM_MEMORY, tools=TOOLS_MEMORY)
    for f in ["Prüfe die Sensordaten von M-07.", "Merke dir bitte den Befund und dass die Elektrofachkraft informiert ist.",
              "Merke dir auch, dass K. Yilmaz heute krank war."]:
        print(f"❓ {f}\n💬 {tag1.frage(f)}\n")
    print("Gespeichert:", lade_memory())

    tag2 = Gespraech(system=SYSTEM_MEMORY, tools=TOOLS_MEMORY)          # neues Gespräch, leerer Verlauf
    f = "Gibt es zu M-07 etwas aus früheren Schichten?"
    print(f"\n— Am nächsten Tag —\n❓ {f}\n💬 {tag2.frage(f)}")

---
# Teil D · Prompt Caching

Viele Anfragen beginnen mit demselben Präfix: Tools, System-Prompt, oft ein langes Dokument. Der Server muss diesen Präfix jedes Mal neu durchrechnen, es sei denn, er speichert die Zwischenergebnisse (den KV-Cache aus Woche 5). Das heißt **Prompt Caching**.
- **Kommerzielle Anbieter** rechnen gecachte Tokens billiger ab, bei Claude Haiku 4.5 etwa ein Zehntel des Input-Preises. Das Schreiben in den Cache kostet dort das 1,25-Fache, und gecacht wird erst ab einer Mindestlänge.
- **Selbst betriebene Server** wie die der KI-Toolbox können Präfixe automatisch wiederverwenden (oft „Prefix Caching“ genannt). Das spart Rechenzeit und damit Wartezeit, Geld spielt für uns keine Rolle. Ob und wie der Dienst das nutzt, sehen wir am Experiment.

Experiment: Wir legen das **ganze Handbuch** in den System-Prompt und stellen fünf Fragen hintereinander. Wir messen die Antwortzeit und lesen, falls der Server sie meldet, die gecachten Tokens aus.

### TODO 8 · Fragen mit langem Präfix
`frage_mit_handbuch(frage)` macht **einen** Aufruf ohne Tools: System-Nachricht `SYSTEM_HANDBUCH`, dann die Frage, `max_tokens=300`. Messt die Dauer mit `time.perf_counter()` und gebt `(antworttext, usage, sekunden)` zurück.

In [ ]:
SYSTEM_HANDBUCH = f"""Du bist der Instandhaltungs-Assistent der Nordwerk GmbH. Beantworte Fragen nur mit dem Handbuch unten und nenne das Kapitel.
Antworte in höchstens drei Sätzen.

<handbuch>
{HANDBUCH}
</handbuch>"""

def frage_mit_handbuch(frage):
    # TODO 8
    pass

In [ ]:
FRAGEN_HANDBUCH = ["Wie oft muss das Spindellager der Drehmaschine geschmiert werden?", "Was bedeutet E33?",
                   "Welchen Netzdruck hat die Druckluft?", "Wo liegt das Ersatz-Spannfutter?", "Wie prüft man eine neue Schleifscheibe?"]
praefix = zaehle(system=SYSTEM_HANDBUCH)
print("System-Prompt mit Handbuch:", praefix, "Tokens")

zeilen = []
if frage_mit_handbuch("Test") is not None:
    for f in FRAGEN_HANDBUCH:
        _, u, sek = frage_mit_handbuch(f)
        zeilen.append({"prompt": u.prompt_tokens, "cached": gecachte_tokens(u), "output": u.completion_tokens, "sekunden": sek})
        print(f"{sek:5.2f} s | Eingabe {u.prompt_tokens:5d} Tokens | davon laut Server aus dem Cache: {gecachte_tokens(u) or 'keine Angabe'}")

    # Was hätte ein kommerzieller Anbieter mit und ohne Prompt Caching berechnet?
    ohne = sum(kosten_cent({"input": z["prompt"], "cache_write": 0, "cache_read": 0, "output": z["output"]}) for z in zeilen)
    mit = sum(kosten_cent({"input": z["prompt"] - praefix, "cache_write": praefix if i == 0 else 0,
                           "cache_read": praefix if i > 0 else 0, "output": z["output"]}) for i, z in enumerate(zeilen))
    print(f"\nKommerziell (Referenz): ohne Caching {ohne:.2f} Cent | mit Caching {mit:.2f} Cent | Ersparnis {1 - mit / ohne:.0%}")

check("TODO 8a Antwort, Nutzung und Dauer", lambda: len(zeilen) == 5 and all(z["sekunden"] > 0 and z["prompt"] > 0 for z in zeilen))
check("TODO 8b Caching spart kommerziell", lambda: mit < ohne)

> **Zum Nachdenken:**
> - Kommt ab der zweiten Frage die Antwort schneller? Dann nutzt der Server vermutlich Prefix Caching. Bleibt alles gleich, ist auch das ein Ergebnis.
> - Kompaktierung und Caching vertragen sich nicht ohne Weiteres: Eine Kompaktierung ändert den Anfang des Verlaufs und macht den Cache ungültig. Wann lohnt sich was?

---
## Bonus

**B1 · Just-in-time statt alles vorab.** Vergleicht die Tokens von Teil D (ganzes Handbuch im Prompt) mit der Suche als Tool (Woche 8) für dieselben fünf Fragen. Was ist schneller, was würde kommerziell mehr kosten?

**B2 · Tool-Ergebnisse kürzen.** Ändert `get_sensor_data` so, dass nur Messgrößen mit `ÜBERSCHRITTEN` zurückkommen, plus die Anzahl unauffälliger. Wie verändert sich der Kontext der Schicht?

**B3 · Ein Jahr Memory.** Was passiert mit `recall`, wenn `memory.json` nach einem Jahr 5.000 Einträge hat? Skizziert eine bessere Lösung (Woche 7).

In [ ]:
# B1 / B2 / B3

## Projekt diese Woche

Im Notebook **`W11_Projekt_Kontext.ipynb`** misst ihr die Kosten eines typischen Gesprächs mit eurem Agenten, baut Kompaktierung, automatisches Caching und ein Langzeit-Memory mit Datenschutzregel ein und vergleicht die **Kosten pro Gespräch vorher und nachher**.

## Zusammenfassung

- Der Kontext ist ein **Budget**: Er kostet Geld und Zeit, und mehr ist nicht automatisch besser.
- Bestandteile: System-Prompt, Tools, Dokumente, Verlauf, Memory. Der Verlauf wächst mit jeder Runde.
- **Kompaktierung** ersetzt alte Runden durch eine Zusammenfassung. Geschnitten wird nur zwischen Fragen.
- **Langzeit-Memory** speichert ausgewählte Notizen über Gespräche hinweg. Was gespeichert werden darf, regelt der **Code**, nicht nur der Prompt, und gespeicherte Einträge müssen löschbar sein.
- **Prompt Caching** verwendet gleichbleibende Präfixe wieder. Kommerziell kostet Cache-Lesen etwa ein Zehntel, auf selbst betriebenen Servern spart es Rechen- und Wartezeit.

### Ausblick auf Woche 12
Unsere Tools stecken im Notebook fest. Wie machen wir sie für andere Agenten, Desktop-Assistenten und das ganze Team nutzbar? **MCP, Frameworks und Multi-Agent.**